# Comparing multiple CaImAn CNMF pipelines

This notebook loads several `.hdf5` CNMF outputs that were run with different parameters...


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from caiman.source_extraction.cnmf.cnmf import load_CNMF

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
MODEL_PATHS = {
    "final": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_final/caiman_results.hdf5",
    "biblio": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio/caiman_results.hdf5",
    "biblio_contoursize": "/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio_contoursize/caiman_results.hdf5",
}



RVAL_THR = 0.8
MIN_SNR = 2.5
MIN_CNN = 0.9


In [ ]:
def load_models(model_paths):
    models = {}
    for name, path in model_paths.items():
        if not os.path.exists(path):
            print(f"Warning: file for '{name}' not found: {path}")
            continue
        print(f"Loading {name} from {path}")
        models[name] = load_CNMF(path)
    return models

def _get_metric(est, candidates, metric_name, model_name):
    """
    Try several possible attribute names and return as np.array.
    Raise a clear error if nothing is found.
    """
    for attr in candidates:
        if hasattr(est, attr):
            val = getattr(est, attr)
            if val is not None:
                return np.array(val)
    raise AttributeError(
        f"{metric_name} not found in Estimates for model '{model_name}'. "
        "Did evaluate_components run correctly?"
    )


def ensure_quality_metrics(cnm, model_name=""):
    """
    Make sure snr / r / cnn metrics exist on cnm.estimates.
    If not, try to recompute them with evaluate_components.
    """
    est = cnm.estimates

    # If we already have any of them, assume evaluate_components has run
    if hasattr(est, "snr_comp") or hasattr(est, "SNR_comp"):
        return

    print(f"Metrics missing for model '{model_name}', trying to run evaluate_components...")

    if cnm.mmap_file is None:
        print(
            f"  -> cnm.mmap_file is None for '{model_name}'. "
            "You need to load your movie as Y and run "
            "cnm.estimates.evaluate_components(Y, cnm.params, dview=None) manually."
        )
        return

    try:
        Yr, dims, T = load_memmap(cnm.mmap_file)
        Y = Yr.T.reshape((T,) + dims, order="F")
        est.evaluate_components(Y, cnm.params, dview=None)
        print("  -> evaluate_components finished.")
    except Exception as e:
        print(f"  -> Could not run evaluate_components for '{model_name}': {e}")


def extract_metrics(cnm, model_name=""):
    """
    Extract quality metrics and accepted / rejected labels from a CNMF object.
    """
    est = cnm.estimates

    # Ensure metrics exist (or at least try)
    ensure_quality_metrics(cnm, model_name=model_name)

    snr = _get_metric(est, ["snr_comp", "SNR_comp"], "SNR", model_name)
    r = _get_metric(est, ["r_values"], "r_values", model_name)
    cnn = _get_metric(est, ["cnn_preds", "cnn_preds_spatial"], "cnn_preds", model_name)

    n_comp = len(snr)
    accepted = np.zeros(n_comp, dtype=bool)
    if hasattr(est, "idx_components"):
        accepted[est.idx_components] = True
    else:
        print(f"Warning: idx_components not found for '{model_name}', assuming all rejected.")
        # all remain False

    return dict(snr=snr, r=r, cnn=cnn, accepted=accepted)


def build_long_dataframe(models):
    """
    For a dict {name: cnm} return a long-form DataFrame with one row per component.
    Columns: model, comp_idx, snr, r, cnn, accepted
    """
    rows = []
    for name, cnm in models.items():
        m = extract_metrics(cnm, model_name=name)
        n = len(m["snr"])
        for i in range(n):
            rows.append(
                dict(
                    model=name,
                    comp_idx=i,
                    snr=m["snr"][i],
                    r=m["r"][i],
                    cnn=m["cnn"][i],
                    accepted=bool(m["accepted"][i]),
                )
            )
    return pd.DataFrame(rows)

In [ ]:
cnm_models = load_models(MODEL_PATHS)
df_all = build_long_dataframe(cnm_models)
display(df_all.head())
summary = summarize_model(df_all)
display(summary)


In [ ]:
def plot_r_vs_snr(df_model, title):
    fig, ax = plt.subplots(figsize=(5.5,4.5))
    rej = df_model[~df_model["accepted"]]
    ax.scatter(rej["r"], rej["snr"], s=10, alpha=0.3, color="gray", label="Rejected")
    acc = df_model[df_model["accepted"]]
    ax.scatter(acc["r"], acc["snr"], s=20, alpha=0.9, color="green", label="Accepted")
    ax.axvline(RVAL_THR, linestyle="--", color="red")
    ax.axhline(MIN_SNR, linestyle="--", color="orange")
    ax.set_xlabel("Spatial corr (r)")
    ax.set_ylabel("SNR")
    ax.set_title(title)
    ax.legend()
    plt.show()

def plot_cnn_vs_snr(df_model, title):
    fig, ax = plt.subplots(figsize=(5.5,4.5))
    rej = df_model[~df_model["accepted"]]
    ax.scatter(rej["cnn"], rej["snr"], s=10, alpha=0.3, color="gray")
    acc = df_model[df_model["accepted"]]
    ax.scatter(acc["cnn"], acc["snr"], s=20, alpha=0.9, color="green")
    ax.axvline(MIN_CNN, linestyle="--", color="red")
    ax.axhline(MIN_SNR, linestyle="--", color="orange")
    ax.set_xlabel("CNN")
    ax.set_ylabel("SNR")
    ax.set_title(title)
    plt.show()

for name, df_m in df_all.groupby("model"):
    plot_r_vs_snr(df_m, f"{name}: r vs SNR")
    plot_cnn_vs_snr(df_m, f"{name}: CNN vs SNR")


In [ ]:
def plot_hist_comparisons(df_all, n_bins=40):
    models = sorted(df_all["model"].unique())
    fig, axes = plt.subplots(1,3, figsize=(14,3.5))
    ax = axes[0]
    for m in models:
        ax.hist(df_all[df_all["model"]==m]["snr"], bins=n_bins, alpha=0.4, label=m)
    ax.set_title("SNR")
    ax = axes[1]
    for m in models:
        ax.hist(df_all[df_all["model"]==m]["r"], bins=n_bins, alpha=0.4)
    ax.set_title("r")
    ax = axes[2]
    for m in models:
        ax.hist(df_all[df_all["model"]==m]["cnn"], bins=n_bins, alpha=0.4)
    ax.set_title("CNN")
    axes[2].legend(models)
    plt.show()

plot_hist_comparisons(df_all)


In [ ]:
# Visual inspection
cn_img = list(cnm_models.values())[0].estimates.Cn

def preview_components_all_models(cnm_models, img, step=5):
    for name, cnm in cnm_models.items():
        print(f"Viewing model: {name}")
        cnm.estimates.nb_view_components(
            img=img,
            idx=cnm.estimates.idx_components[::step],
            cmap="gray",
            denoised_color="red"
        )

preview_components_all_models(cnm_models, cn_img, step=5)
